In [84]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix
)

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    BatchNormalization,
    Dropout
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

In [85]:
df = pd.read_csv(
    "../data/processed/multiclass_balanced.csv"
)

print(df.shape)

(856509, 81)


In [86]:
df["Anomaly"] = (
    df["AttackFamily"] != "BENIGN"
).astype(int)

df["Anomaly"].value_counts()

Anomaly
1    556509
0    300000
Name: count, dtype: int64

In [87]:
X = df.drop(
    [
        "Label",
        "AttackFamily",
        "Target",
        "Anomaly"
    ],
    axis=1
)

y = df["Anomaly"]

In [88]:
X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.fillna(0)

X = X.astype("float32")

In [89]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [90]:
X_train = X_scaled[
    y == 0
]

print(X_train.shape)

(300000, 78)


In [91]:
input_dim = X_train.shape[1]

input_layer = Input(
    shape=(input_dim,)
)

x = Dense(
    128,
    activation="relu"
)(input_layer)

x = BatchNormalization()(x)

x = Dropout(0.2)(x)

x = Dense(
    64,
    activation="relu"
)(x)

x = Dense(
    32,
    activation="relu"
)(x)

encoded = Dense(
    16,
    activation="relu"
)(x)

x = Dense(
    32,
    activation="relu"
)(encoded)

x = Dense(
    64,
    activation="relu"
)(x)

x = Dense(
    128,
    activation="relu"
)(x)

decoded = Dense(
    input_dim,
    activation="linear"
)(x)

autoencoder = Model(
    input_layer,
    decoded
)

autoencoder.compile(
    optimizer="adam",
    loss="mse"
)

autoencoder.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 78)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_46 (Dense)                │ (None, 128)            │        10,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_47 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_48 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_49 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_50 (Dense)                │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_51 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_52 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_53 (Dense)                │ (None, 78)             │        10,062 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 42,526 (166.12 KB)

 Trainable params: 42,270 (165.12 KB)

 Non-trainable params: 256 (1.00 KB)

In [92]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3
)

history = autoencoder.fit(
    X_train,
    X_train,
    epochs=100,
    batch_size=1024,
    validation_split=0.1,
    shuffle=True,
    callbacks=[
        early_stop,
        reduce_lr
    ]
)

Epoch 1/100
264/264 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.7463 - val_loss: 0.4202 - learning_rate: 0.0010
Epoch 2/100
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.4323 - val_loss: 0.2684 - learning_rate: 0.0010
Epoch 3/100
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.3556 - val_loss: 0.3991 - learning_rate: 0.0010
Epoch 4/100
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.3344 - val_loss: 3.9115 - learning_rate: 0.0010
Epoch 5/100
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.2744 - val_loss: 1.5230 - learning_rate: 0.0010
Epoch 6/100
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1949 - val_loss: 1.4286 - learning_rate: 5.0000e-04
Epoch 7/100
264/264 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.1455 - val_loss: 0.4386 - learning_rate: 5.0000e-04
Epoch 8/100
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.1479 - val_loss: 2.1002 - learning_rate: 5.0000e-04
Epoch 9/100
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.1596 - val_loss: 1.5279 - learning_r

In [93]:
reconstructions = autoencoder.predict(
    X_scaled,
    batch_size=2048
)

419/419 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step


In [94]:
mse = np.mean(
    np.square(
        X_scaled - reconstructions
    ),
    axis=1
)

In [95]:
train_recon = autoencoder.predict(
    X_train,
    batch_size=2048
)

train_mse = np.mean(
    np.square(
        X_train - train_recon
    ),
    axis=1
)

print(train_mse.min())
print(train_mse.max())
print(train_mse.mean())

147/147 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
0.0034515262
17360.104
0.42475307


In [99]:
from sklearn.ensemble import IsolationForest

iso_50 = IsolationForest(
    n_estimators=300,
    contamination=0.50,
    random_state=42,
    n_jobs=-1
)

iso_50.fit(X_train)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",300
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=42)


In [100]:
for p in [95,97,98,99,99.5]:

    threshold = np.percentile(
        train_mse,
        p
    )

    preds = (
        mse > threshold
    ).astype(int)

    print("\nPercentile:", p)

    print(
        classification_report(
            y,
            preds
        )
    )


Percentile: 95
              precision    recall  f1-score   support

           0       0.46      0.95      0.62    300000
           1       0.94      0.40      0.56    556509

    accuracy                           0.59    856509
   macro avg       0.70      0.68      0.59    856509
weighted avg       0.77      0.59      0.58    856509


Percentile: 97
              precision    recall  f1-score   support

           0       0.44      0.97      0.60    300000
           1       0.95      0.33      0.49    556509

    accuracy                           0.55    856509
   macro avg       0.70      0.65      0.54    856509
weighted avg       0.77      0.55      0.53    856509


Percentile: 98
              precision    recall  f1-score   support

           0       0.40      0.98      0.57    300000
           1       0.95      0.21      0.34    556509

    accuracy                           0.48    856509
   macro avg       0.67      0.59      0.45    856509
weighted avg       0.76   

In [101]:
import joblib

joblib.dump(
    iso_50,
    "../models/isolation_forest_v1.pkl"
)

joblib.dump(
    scaler,
    "../models/anomaly_scaler.pkl"
)

['../models/anomaly_scaler.pkl']

In [102]:
import os

os.listdir("../models")

['.gitkeep',
 'anomaly_scaler.pkl',
 'attack_encoder.pkl',
 'attack_mapping.pkl',
 'ddos_xgb_v1.pkl',
 'isolation_forest_v1.pkl',
 'multiclass_xgb_v1.pkl']